In [2]:
import pandas as pd

orders       = pd.read_csv("https://cdn.enqurious.com/documents/0518fd79-992c-420c-8580-a7acf31172b6_exorders.csv", parse_dates=["order_purchase_date"])
transactions = pd.read_csv("https://cdn.enqurious.com/documents/c1b6bff1-a6af-431b-bcc8-9618990bc5df_extransactions.csv")
products     = pd.read_csv("https://cdn.enqurious.com/documents/604aa913-c904-4e6c-9d26-fdd443c52482_exproducts.csv")


In [3]:
orders.head(2)

,order_id,customer_id,partner_id,ship_mode,order_status,order_purchase_date,order_approved_at,order_dispatched_date,order_delivered_date,order_estimated_delivery_date
0,CA-2014-100006,MH-17785,VEN02,Standard,delivered,2018-05-11 20:07:00,2018-05-11 20:30:00.000,2018-05-16 15:14:00.000,2018-05-19 13:42:00.000,2018-05-23
1,CA-2014-100090,PC-18745,VEN01,Standard,delivered,2018-01-30 10:21:00,2018-01-30 10:35:00.000,2018-01-31 20:39:00.000,2018-02-14 23:39:00.000,2018-02-26


In [4]:
products.head(2)

,product_id,product_name,colors,category,sub_category,date_added,manufacturer,sizes,upc,weight,product_photos_qty
0,FUR-BO-10004357,O'Sullivan Living Dimensions 3-Shelf Bookcases,White,Furniture,Bookcases,2016-06-08,Dearfoams,NaN,39161445658,NaN,8
1,FUR-CH-10002044,Office Star - Contemporary Task Swivel chair w...,Blue,Furniture,Chairs,2017-01-09,Easy Spirit,8.5,29021040932,NaN,2


In [5]:
transactions.head(2)

,TransactionID,Order_ID,Product_ID,Sales_Amount,Quantity,Discount,COGS
0,1,CA-2014-145317,FUR-BO-10001798,261.96,2,0.0,41.9136
1,2,CA-2016-118689,FUR-CH-10000454,731.94,3,0.0,219.5820


In [6]:
def calculate_sales(order_transactions_product_df, product_category=None, **kwargs):

    start_date = kwargs.get('start_date')
    end_date = kwargs.get('end_date')

    today = pd.Timestamp.today().normalize()

    # Case 1: Neither date provided
    if start_date is None and end_date is None:
        end_date = today
        start_date = end_date - pd.Timedelta(days=30)

    # Case 2: Only end_date provided
    elif start_date is None:
        end_date = pd.to_datetime(end_date)
        start_date = end_date - pd.Timedelta(days=30)

    # Case 3: Only start_date provided
    elif end_date is None:
        start_date = pd.to_datetime(start_date)
        end_date = today

    # Case 4: Both dates provided
    else:
        start_date = pd.to_datetime(start_date)
        end_date = pd.to_datetime(end_date)

    # Filter by date
    filtered_df = order_transactions_product_df[
        (order_transactions_product_df['order_purchase_date'] >= start_date) &
        (order_transactions_product_df['order_purchase_date'] <= end_date)
    ]

    # Optional category filter
    if product_category is not None:
        filtered_df = filtered_df[
            filtered_df['category'] == product_category
        ]

    # Total sales
    total_sales = filtered_df['Sales_Amount'].sum()

    return total_sales

In [ ]:
total_sales = calculate_sales(
    order_transactions_product_df,
    start_date='2016-10-01',
    end_date='2017-10-01'
)
